# Ofensivo / No Ofensivo (es + en) — afinamiento en Google Colab

Este notebook corre el mismo código que está en el repo (`Ofensivo_o_NoOfensivo/`), pero aquí con GPU gratuita de Colab.

**Antes de empezar:** en el menú de arriba ve a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución` y selecciona **GPU** (T4 es suficiente, es la gratuita).

**Sí se necesita una cuenta de Hugging Face** (gratis) para este notebook, a diferencia de Traductor: una de las 4 fuentes de datos (OffendES) está *gated* — hay que iniciar sesión y aceptar sus condiciones en su página ANTES de poder descargarla con código. Ver Parte 3.

## 1. Obtener el código del proyecto

In [ ]:
REPO = "Gad-gg-23/Ofencivo_O_No"

import os
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull

%cd /content/repo

## 2. Instalar lo que falta y verificar GPU

Colab ya trae preinstalados PyTorch (con CUDA), pandas y scikit-learn. **No reinstales torch** (podrías romper la versión con CUDA que ya trae Colab) — los demás sí se actualizan juntos en una sola instalación, para que sus versiones queden alineadas entre sí.

In [ ]:
!pip install -q -U transformers datasets accelerate huggingface_hub

import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU activa. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

## 3. Iniciar sesión en Hugging Face (necesario por OffendES)

1. Si todavía no lo hiciste: entra a https://huggingface.co/datasets/SINAI/OffendES mientras tienes sesión iniciada, y acepta las condiciones de acceso (botón en la página). Solo se hace una vez por cuenta.
2. Corre la celda de abajo y pega tu token (https://huggingface.co/settings/tokens, basta con permiso **Read**).

In [ ]:
from huggingface_hub import login
login()

## 4. Descargar y preparar los datos

Se vuelve a hacer cada vez porque Colab reinicia la sesión seguido y borra todo lo descargado. `preparar` incluye generar las frases sintéticas de code-switching (es+en mezclados).

In [ ]:
!python Ofensivo_o_NoOfensivo.py descargar

In [ ]:
!python Ofensivo_o_NoOfensivo.py preparar

## 5. Baseline rápido (TF-IDF + Regresión Logística)

Corre en CPU en segundos. Sirve como referencia para saber si el Transformer realmente aporta algo, no solo para tener "el modelo más grande".

In [ ]:
!python Ofensivo_o_NoOfensivo.py entrenar-base

## 6. Detectores de categoría (racismo, clasismo, etc.)

Un modelo chico por categoría (TF-IDF + Regresión Logística), entrenado solo con la fuente que trae esa etiqueta (`piuba_odio` para las 8 de odio hacia un grupo, `OffendES` para insulto_personal). Alimentan el "comité de categorías" del ensamble (Parte 9). Corre en CPU en segundos.

In [ ]:
!python Ofensivo_o_NoOfensivo.py entrenar-categorias

## 7. Afinar el Transformer (XLM-RoBERTa preentrenado)

Parte de `xlm-roberta-base` (ya sabe de ~100 idiomas) y se afina con nuestros datos en español, inglés y frases mixtas. Tarda varios minutos en GPU — va a imprimir el progreso por época. **Espera a que termine por completo** antes de avanzar a la siguiente celda.

In [ ]:
!python Ofensivo_o_NoOfensivo.py entrenar

## 8. Evaluar: baseline vs. Transformer vs. ensamble (incluye validación y code-switching)

Accuracy/Precision/Recall/F1 de los 3 (baseline, Transformer, y el ensamble por votación de ambos + comité de categorías) sobre `prueba.tsv`, con intervalo de confianza bootstrap del 95% sobre el resultado general, desglosado por idioma y por fuente, más el desempeño sobre `prueba_codeswitch.tsv` (frases que mezclan español e inglés, nunca vistas en entrenamiento como mezcla real — solo sus mitades por separado).

In [ ]:
!python Ofensivo_o_NoOfensivo.py evaluar

In [ ]:
import pandas as pd

tabla = pd.read_csv("salidas/resultados_comparacion.csv")

print("=== General (prueba.tsv), con intervalo de confianza del 95% ===")
columnas = ["modelo", "subconjunto", "accuracy", "accuracy_ci_baja", "accuracy_ci_alta",
            "f1_ofensivo", "f1_ofensivo_ci_baja", "f1_ofensivo_ci_alta"]
general = tabla[tabla.subconjunto == "general"]
print(general[[c for c in columnas if c in general.columns]].to_string(index=False))

print("\n=== Por idioma ===")
print(tabla[tabla.subconjunto.str.startswith("idioma=")].to_string(index=False))

print("\n=== Por fuente ===")
print(tabla[tabla.subconjunto.str.startswith("fuente=")].to_string(index=False))

print("\n=== Code-switching (frases mixtas es+en) ===")
print(tabla[tabla.subconjunto.str.startswith("codeswitch_")].to_string(index=False))

## 10. Descargar el modelo afinado o subirlo a Hugging Face (opcional)

Los pesos de XLM-RoBERTa afinado pesan **~1.1 GB** — no caben en GitHub (límite de 100 MB por archivo), por eso se comparten en Hugging Face en vez de en el repo, igual que el modelo de `Traductor`.

**Opción A — descargarlo a tu computadora** (como .zip):

In [ ]:
!zip -r -q modelo_xlmr.zip salidas/modelo_xlmr
from google.colab import files
files.download("modelo_xlmr.zip")

**Opción B — subirlo a Hugging Face** (recomendado, así el equipo comparte el último modelo entrenado). Necesitas un *access token* con permiso de escritura:

1. Ve a https://huggingface.co/settings/tokens
2. **Create new token** → tipo **Write** → nómbralo (ej. "Colab Ofensivo") → **Create token**.
3. Cópialo — solo se muestra una vez.

A diferencia de GitHub, **no hace falta que alguien cree el repositorio de antemano**: `push_to_hub` lo crea solo la primera vez (bajo el repo `config.HF_REPO_MODELO`). Si tu usuario de Hugging Face no es colaborador de ese repo, pídele al admin que te agregue desde Settings → Collaborators del repo en huggingface.co.

In [ ]:
import getpass
from huggingface_hub import create_repo

import config

HF_TOKEN = getpass.getpass("Tu Hugging Face access token (permiso Write): ")

create_repo(config.HF_REPO_MODELO, token=HF_TOKEN, exist_ok=True)

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained(config.RUTA_MODELO_DIR)
tokenizer = AutoTokenizer.from_pretrained(config.RUTA_MODELO_DIR)
modelo.push_to_hub(config.HF_REPO_MODELO, token=HF_TOKEN)
tokenizer.push_to_hub(config.HF_REPO_MODELO, token=HF_TOKEN)

print(f"OK: modelo subido a https://huggingface.co/{config.HF_REPO_MODELO}")

## 11. Probar con tu propia oración (ensamble por votación)

Usa el ensamble de `votar.py` (baseline + Transformer + comité de categorías) y muestra el detalle de los 3 votos. Prueba con español, inglés, o mezclando ambos (modismos/code-switching).

In [ ]:
TEXTO = "eres un idiota, you're such a loser"

!python votar.py --texto "{TEXTO}"

## 12. Interpretabilidad: ¿qué palabras influyeron más? (`--explicar`)

`predecir.py --explicar` usa el Transformer y hace *leave-one-out*: quita cada palabra, una a la vez, y mide cuánto cae la probabilidad de "ofensivo" al quitarla.

In [ ]:
!python predecir.py --texto "{TEXTO}" --explicar